### Type Annotation Handling

In [1]:
from __future__ import annotations

### Implement each half (a-n, o-z, A-M, N-Z, 0-9) as its own closed cyclic range
- so a shift can never push a character out of its own category.
- to guarantee that encrypt_file() and decrypt_file() are always exact inverses of one another, no matter which non-negative integers are supplied for shift1 and shift2.

In [2]:
def _shift_within_range(char: str, start: str, size: int, amount: int) -> str:
    """Cyclically shift `char` by `amount` positions within a contiguous
    range of `size` characters that begins at `start`.

    A positive `amount` shifts forward (later in the alphabet/digits);
    a negative `amount` shifts backward.
    """
    offset = ord(char) - ord(start)
    new_offset = (offset + amount) % size
    return chr(ord(start) + new_offset)

### Shift rules (applied per character, based on the character's category):
- Lowercase a-n : shift forward within {a..n} by (shift1 * shift2)
- Lowercase o-z : shift backward within {o..z} by (shift1 + shift2)
- Uppercase A-M : shift backward within {A..M} by shift1
- Uppercase N-Z : shift forward within {N..Z} by shift2 ** 2
- Digits   0-9  : shift forward within {0..9} by (shift1 - shift2)
- Anything else : left unchanged (spaces, tabs, newlines, punctuation, symbols)

In [3]:

def _transform_char(char: str, shift1: int, shift2: int, *, encrypting: bool) -> str:
    """Apply the cipher rule for a single character.

    The same rule table is used for both directions: `encrypting=True`
    applies the forward (encryption) shift amounts, `encrypting=False`
    applies the exact negation of those amounts, which undoes them.
    """
    sign = 1 if encrypting else -1

    if 'a' <= char <= 'n':
        return _shift_within_range(char, 'a', 14, sign * (shift1 * shift2))
    if 'o' <= char <= 'z':
        return _shift_within_range(char, 'o', 12, -sign * (shift1 + shift2))
    if 'A' <= char <= 'M':
        return _shift_within_range(char, 'A', 13, -sign * shift1)
    if 'N' <= char <= 'Z':
        return _shift_within_range(char, 'N', 13, sign * (shift2 ** 2))
    if char.isdigit():
        return _shift_within_range(char, '0', 10, sign * (shift1 - shift2))

    return char